In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from scipy.stats import uniform, randint
import json
import warnings
warnings.filterwarnings('ignore')

print("--- STARTING NEW MODEL TRAINING (7 FEATURES) ---")

# --- 1. Load Cleaned Data ---
# Make sure this file is in the same folder as your notebook
try:
    df_freq = pd.read_csv('/Users/senuja/Jupyter Notebook/Video_Game_Sales_Predictor/results/outputs/Group Pipeline (Frequency).csv')
    print("Loaded 'Group Pipeline (Frequency).csv'")
except Exception as e:
    print(f"Error: Could not load 'Group Pipeline (Frequency).csv'. {e}")
    print("Please make sure it's in the same folder as this notebook.")
    # Stop execution if file isn't found
    raise e

# --- 2. Define the 7 Features Your Website Uses ---
# This is the most important change!
# We are selecting ONLY the features your website form can provide.
feature_cols = [
    'Critic Score',
    'Release Year',
    'Publisher Freq',
    'Developer Freq',
    'Console Freq',
    'Genre Freq',
    'Game Quality Encoded'
]

print(f"Training will use ONLY these 7 features: {feature_cols}")

# --- 3. Define X and y ---
# Check if all columns exist
missing_cols = [col for col in feature_cols if col not in df_freq.columns]
if missing_cols:
    print(f"Error: Your 'Group Pipeline (Frequency).csv' is missing columns: {missing_cols}")
    raise Exception("Missing required columns in CSV.")

y = np.log1p(df_freq['Sales']) # Use log-transformed target
X = df_freq[feature_cols]      # Use ONLY the 7 features

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 4. Re-Train and Tune the NEW XGBoost Model ---
print("Training the new 7-feature XGBoost model...")
# (We use the same tuning parameters as your successful model)
param_dist = {
    'n_estimators': randint(100, 500),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(3, 10)
}
xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)
random_search = RandomizedSearchCV(
    xgb_model, 
    param_distributions=param_dist, 
    n_iter=25, 
    cv=5, 
    random_state=42, 
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)
random_search.fit(X_train, y_train)

# This is your new, final model
best_xgb_new = random_search.best_estimator_
print(f"New model trained with parameters: {random_search.best_params_}")

# --- 5. Save the NEW Model ---
best_xgb_new.save_model("xgb_model_final.json")
print("\nSUCCESS: New model saved to 'xgb_model_final.json'")

# --- 6. Create and Save Feature Maps (from Raw Data) ---
# This part builds the "translation dictionary"
print("Loading 'vgchartz-2024.csv' to build feature maps...")
try:
    # Make sure this file is also in the same folder
    df_raw = pd.read_csv('/Users/senuja/Jupyter Notebook/Video_Game_Sales_Predictor/data/raw/vgchartz-2024.csv')
except Exception as e:
    print(f"Error: Could not load 'vgchartz-2024.csv'. {e}")
    raise e

feature_maps = {
    "publisher": df_raw['publisher'].value_counts(normalize=True).to_dict(),
    "developer": df_raw['developer'].value_counts(normalize=True).to_dict(),
    "console": df_raw['console'].value_counts(normalize=True).to_dict(),
    "genre": df_raw['genre'].value_counts(normalize=True).to_dict(),
    "quality": { "High": 2, "Medium": 1, "Low": 0 }
}
with open('feature_maps_final.json', 'w') as f:
    json.dump(feature_maps, f)
print("SUCCESS: Feature maps saved to 'feature_maps_final.json'")

# --- 7. Save the NEW 7-Feature Order ---
# This saves the list of the 7 features
feature_order_new = list(X.columns)
with open('feature_order_final.json', 'w') as f:
    json.dump(feature_order_new, f)
print("SUCCESS: New feature order saved to 'feature_order_final.json'")
print(f"New feature order is: {feature_order_new}")
print("\n--- All 3 new files are ready for upload to Replit! ---")

--- STARTING NEW MODEL TRAINING (7 FEATURES) ---
Loaded 'Group Pipeline (Frequency).csv'
Training will use ONLY these 7 features: ['Critic Score', 'Release Year', 'Publisher Freq', 'Developer Freq', 'Console Freq', 'Genre Freq', 'Game Quality Encoded']
Training the new 7-feature XGBoost model...
New model trained with parameters: {'learning_rate': np.float64(0.05680559213273095), 'max_depth': 5, 'n_estimators': 314}

SUCCESS: New model saved to 'xgb_model_final.json'
Loading 'vgchartz-2024.csv' to build feature maps...
SUCCESS: Feature maps saved to 'feature_maps_final.json'
SUCCESS: New feature order saved to 'feature_order_final.json'
New feature order is: ['Critic Score', 'Release Year', 'Publisher Freq', 'Developer Freq', 'Console Freq', 'Genre Freq', 'Game Quality Encoded']

--- All 3 new files are ready for upload to Replit! ---
